In [1]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Aave V3.1 — normalization stage (raw → real units)

The `optimized_queries` now emit **raw** values (no in-SQL division), so normalization lives
**here**, in one place, between the API fetch and `transform.ipynb`:

    optimized_queries (raw) → API fetch → THIS notebook → transform.ipynb

`transform.ipynb` consumes already-normalized frames (its own per-table scaling is removed).

**Scope** = the tables whose normalization used to happen in SQL (`flashloan`,
`reserve_state_rates`, `borrow_repay` amounts) **plus** `reserve_config` (moved out of transform).

Because tables are re-fetched **independently**, whether a given CSV is raw is decided **per table
by a guard on the data itself** (`looks_raw_bps`, `looks_raw_ray`, `looks_raw_token_amounts`), not
by a global flag. Every cell is therefore **idempotent** — run this notebook after *any* fetch and
only the still-raw tables are touched.

Logic lives in `normalization.py`; results are written back to `query_result_data/` (overwrite).

In [ ]:
import pandas as pd
import normalization as nz
import transform as tf

DATA_DIR = "query_result_data"

def latest(pattern):
    """Newest versioned CSV matching e.g. 'reserve_config_*.csv'.

    Delegates to transform.newest_matching rather than sorted(glob(...))[-1]. Filenames
    now carry the DATE WINDOW instead of the fetch time, and a plain lexicographic sort
    ranks '..._2025-04-01_2026-03-31.csv' BELOW a legacy '..._20260822T191557Z.csv'
    (they differ at the 4th char, '5' < '6') — so the old file would silently keep
    winning here while transform.load_table resolved the new one. One resolver, one
    answer."""
    path = tf.newest_matching(pattern, DATA_DIR)
    if path is None:
        raise FileNotFoundError(f"no CSV matching {pattern!r} in {DATA_DIR}/")
    return str(path)

## reserve_config — raw bps / 2dp → real units

Scales `debt_ceiling` (÷1e2) and `reserve_factor` / `liquidation_threshold` /
`liquidation_bonus` / `ltv` (÷1e4) from the **part-4 decimal reference**; `supply_cap` /
`borrow_cap` are whole tokens (decimals 0) so they stay as-is. This reproduces exactly what
`transform.ipynb` used to do in `scale_columns_by_decimals`, so downstream values are unchanged.
The `looks_raw_bps` guard makes the cell **idempotent** (re-running on scaled data is a no-op).

In [3]:
cfg_path = latest("reserve_config_*.csv")
df_config = pd.read_csv(cfg_path)
df_ref    = pd.read_csv(latest("decimal_reference_part4_*.csv"))

preview = ["supply_cap", "debt_ceiling", "reserve_factor", "liquidation_threshold", "ltv"]
print("loaded :", cfg_path.split("/")[-1], "->", df_config.shape)
print("raw    :", df_config[preview].iloc[0].to_dict())

if nz.looks_raw_bps(df_config["ltv"]):
    df_config = nz.normalize_reserve_config(df_config, df_ref)
    df_config.to_csv(cfg_path, index=False)          # overwrite in place (query_result_data/)
    print("action : normalized & overwrote", cfg_path.split("/")[-1])
else:
    print("action : already normalized -- skipped (idempotent)")

print("result :", df_config[preview].iloc[0].to_dict())

loaded : reserve_config_7804264_20260725T094252Z.csv -> (232879, 11)
raw    : {'supply_cap': 9000000.0, 'debt_ceiling': 4500000.0, 'reserve_factor': 0.2, 'liquidation_threshold': 0.67, 'ltv': 0.57}
action : already normalized -- skipped (idempotent)
result : {'supply_cap': 9000000.0, 'debt_ceiling': 4500000.0, 'reserve_factor': 0.2, 'liquidation_threshold': 0.67, 'ltv': 0.57}


## flashloan / reserve_state / borrow_repay — auto-detected per table

These three are normalized **only if their current CSV still looks raw**, decided per table by a
guard instead of a hand-set flag. The old `RE_FETCHED = True/False` switch was all-or-nothing, so a
**partial** re-fetch broke it either way: leaving it `False` skips a genuinely raw table, flipping it
`True` double-scales the ones already in real units. That is exactly what happened on 2026-07-25 —
only `borrow_repay` was re-fetched raw, the flag stayed `False`, and un-scaled wei amounts flowed
into `DF_common_final` (caught downstream by the Tier-1 hard-fail gate in `validation.ipynb`).

Guards: `looks_raw_ray` (RAY rates / indexes still ≫ 1) and `looks_raw_token_amounts` (amounts still
on the token-decimals scale — the same `|v| >= 1e15` floor `adv_validation`'s Tier-1 *un-scaled wei
magnitude* check uses, so normalization and validation agree on what "raw" means).

Token decimals for the amount tables come from the oracle-price table's `decimals` column (per
asset) — the same source `transform.ipynb` used. The cell is **idempotent**: re-running it on
already-normalized CSVs is a no-op.

In [4]:
# Normalize each table only if its CSV still looks raw (per-table guard -> idempotent).
oracle = pd.read_csv(latest("oracle_price_usd_eth_weth_6h_*.csv"))
token_decimals = dict(zip(oracle["asset"], oracle["decimals"]))

# (file pattern, columns to scale, guard kind) -- reserve_state is RAY, the rest are token amounts
TABLES = [
    ("reserve_state_rates_*.csv",
     ["liquidity_rate", "variable_borrow_rate", "liquidity_index", "variable_borrow_index"], "ray"),
    ("flashloan_*.csv",    ["flashloan_amount", "flashloan_premium"],           "token"),
    ("borrow_repay_*.csv", ["borrow_amount", "repay_amount", "net_debt_flow"],  "token"),
]

for pattern, cols, kind in TABLES:
    path = latest(pattern)
    df_t = pd.read_csv(path)
    peek = [c for c in cols if c in df_t.columns]
    is_raw = (nz.looks_raw_ray(df_t[peek[0]]) if kind == "ray"
              else nz.looks_raw_token_amounts(df_t, cols))
    before = {c: float(pd.to_numeric(df_t[c], errors="coerce").abs().max()) for c in peek}
    name = path.split("/")[-1]

    if not is_raw:
        print(f"{name:55s} already normalized -- skipped")
        continue

    out = (nz.normalize_reserve_state(df_t) if kind == "ray"
           else nz.normalize_amounts_by_token_decimals(df_t, token_decimals, cols))
    out.to_csv(path, index=False)                     # overwrite in place (query_result_data/)
    after = {c: float(pd.to_numeric(out[c], errors="coerce").abs().max()) for c in peek}
    print(f"{name:55s} RAW -> normalized & overwrote")
    print("    max before:", {k: f"{v:.4g}" for k, v in before.items()})
    print("    max after :", {k: f"{v:.4g}" for k, v in after.items()})

reserve_state_rates_7711042_20260624T074536Z.csv        already normalized -- skipped
flashloan_7798349_20260624T082343Z.csv                  already normalized -- skipped
borrow_repay_7798273_20260725T094440Z.csv               RAW -> normalized & overwrote
    max before: {'borrow_amount': '6.441e+26', 'repay_amount': '6.538e+26', 'net_debt_flow': '5.46e+26'}
    max after : {'borrow_amount': '6.441e+08', 'repay_amount': '6.538e+08', 'net_debt_flow': '5.46e+08'}
